# Predicción de Churn en Telco — 03 · Entrenamiento del modelo final

Entrena y guarda el modelo final seleccionado en `02_experimentos.ipynb`:
**LightGBM** sobre features base (one-hot + estandarización), con los hiperparámetros
encontrados por `RandomizedSearchCV` (PR-AUC 0,672 en CV 5 folds estratificada).

Al ejecutarse de punta a punta este notebook replica exactamente el modelo final:
produce `modelo_final.joblib` (pipeline completo) y `umbral_operativo.json`
(umbral de decisión elegido por F2 sobre predicciones out-of-fold del train).

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

SEED = 42

def cargar_datos():
    """Carga y limpieza minima definida en 01_eda.ipynb."""
    df = pd.read_csv("WA_Fn-UseC_-Telco-Customer-Churn.csv")
    df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce").fillna(0)
    X = df.drop(columns=["customerID", "Churn"])
    y = (df["Churn"] == "Yes").astype(int)
    ids = df["customerID"]
    return X, y, ids

def particion(X, y, ids):
    """Misma particion estratificada 80/20 usada en 02_experimentos.ipynb."""
    return train_test_split(X, y, ids, test_size=0.20, stratify=y, random_state=SEED)

In [2]:
import json
import joblib
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import precision_recall_curve
from lightgbm import LGBMClassifier

# Hiperparametros del ganador de la busqueda aleatoria (02_experimentos.ipynb)
PARAMS_LGBM = dict(
    colsample_bytree=0.6296178606936361,
    learning_rate=0.0338452204120114,
    min_child_samples=50,
    n_estimators=127,
    num_leaves=13,
    reg_alpha=2.512779099948728,
    reg_lambda=0.0627773091695805,
    scale_pos_weight=1.0,
    subsample=0.6254233401144095,
    subsample_freq=1,
)

def construir_pipeline():
    prep = ColumnTransformer([
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False),
         make_column_selector(dtype_include=object)),
        ("num", StandardScaler(), make_column_selector(dtype_include=np.number)),
    ])
    return Pipeline([("prep", prep),
                     ("clf", LGBMClassifier(random_state=SEED, verbose=-1,
                                            n_jobs=1, **PARAMS_LGBM))])

In [3]:
X, y, ids = cargar_datos()
X_train, X_test, y_train, y_test, ids_train, ids_test = particion(X, y, ids)

# Umbral operativo: maximiza F2 sobre predicciones out-of-fold del train
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
scores_oof = cross_val_predict(construir_pipeline(), X_train, y_train,
                               cv=cv, method="predict_proba", n_jobs=-1)[:, 1]
prec, rec, thr = precision_recall_curve(y_train, scores_oof)
f2 = (5 * prec * rec) / np.where((4 * prec + rec) == 0, np.nan, 4 * prec + rec)
umbral = float(thr[np.nanargmax(f2[:-1])])

# Modelo final: entrenado sobre todo el train
modelo_final = construir_pipeline()
modelo_final.fit(X_train, y_train)

joblib.dump(modelo_final, "modelo_final.joblib")
with open("umbral_operativo.json", "w") as f:
    json.dump({"umbral": umbral, "criterio": "F2 maximo (OOF, CV 5 folds)"}, f, indent=2)

print(f"Modelo guardado: modelo_final.joblib")
print(f"Umbral operativo (F2): {umbral:.4f} -> umbral_operativo.json")

Modelo guardado: modelo_final.joblib
Umbral operativo (F2): 0.1526 -> umbral_operativo.json
